In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [3]:
# def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
#     i_max = 0
#     alpha_max = 0
#     val_max = -np.inf

#     for i in range(theta_0.shape[0]):
#     # for i in range(X_0.shape[1]):
#         theta_r_min = deepcopy(theta_0)
#         theta_r_max = deepcopy(theta_0)
        
#         theta_r_min[i] -= alpha
#         theta_r_max[i] += alpha
#         weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
#         weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
#         J_min, J_max = [], []
#         for xi in range(len(X_r)):
#             x_0 = X_0[xi]
#             x_r = X_r[xi]
#             J = RecourseCost(x_0, lamb)
#             j_min = J.eval(x_r, weights_r_min, bias_r_min)
#             j_max = J.eval(x_r, weights_r_max, bias_r_max)
#             J_min.append(j_min)
#             J_max.append(j_max)
        
#         if np.mean(J_min) > np.mean(J_max):
#             if np.mean(J_min) > val_max:
#                 i_max = i
#                 alpha_max = -alpha
#                 val_max = np.mean(J_min).item()
#         else:
#             if np.mean(J_max) > val_max:
#                 i_max = i
#                 alpha_max = alpha
#                 val_max = np.mean(J_max).item()
    
#     theta_adv = deepcopy(theta_0)
#     theta_adv[i_max] += alpha_max
#     return theta_adv

In [4]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [5]:
def get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb):
    
    theta_adv = deepcopy(theta_0)
    
    for i in range(theta_0.shape[0]):
        theta_r_min = deepcopy(theta_0)
        theta_r_max = deepcopy(theta_0)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        
        if np.mean(J_min) > np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha

    return theta_adv

In [6]:
def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
    thetas = generateThetas(theta_0, alpha)
    if alpha == 0:
        return thetas[0].copy()
    
    Js = np.empty((X_0.shape[0], thetas.shape[0]))
    for i in range(X_0.shape[0]):
         J = RecourseCost(X_0[i], lamb)
         for j, theta in enumerate(thetas):  
            Js[i, j] = J.eval(X_r[i], theta[:-1], np.array([theta[-1]]))
    
    Js_sum = Js.sum(axis=0) 
    Js_sum_maxI = np.argmax(Js_sum)
    theta_adv = thetas[Js_sum_maxI]

    return theta_adv

def generateThetas(theta0 : np.ndarray, alpha):
        # theta0 has bias
        thetas = theta0.copy()
        if alpha == 0:
            return np.array([thetas])
        
        thetas = np.repeat(thetas.reshape(1, theta0.size), (theta0.size * 2) - 1, axis=0)
        thetas_i = 0

        for i in range(theta0.size):
            if i == theta0.size - 1:
                thetas[thetas_i][i] -= alpha
                thetas_i += 1
                break

            thetas[thetas_i][i] += alpha
            thetas_i += 1
            thetas[thetas_i][i] -= alpha
            thetas_i += 1

        return thetas

In [7]:
def evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]

    if alpha != 0:
        if theta_adv_method=='L-1':
            theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
        else:
            theta_adv = get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = theta_0.copy()
    
    weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    
    clf_adv = deepcopy(clf)
    clf_adv.model.coef_ = weights_adv.reshape(1,-1)
    clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [8]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    # xP has bias
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, bias_adv))

def calTheta(xP: np.array, weights: np.array, bias: np.array, alpha: float, methods: str):
    if methods == "L-inf":
        thetaP = calThetaAdv_linf(xP, weights, bias, alpha)
    else:
        thetaP = calThetaAdv_l1(np.hstack((xP, np.array([1]))), np.hstack((weights, bias)), alpha)

    return thetaP[:-1], np.array([thetaP[-1]])

In [9]:
def evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]
    # Don't need this block of code below
    if theta_adv_method=='L-1':
        theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb)
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0    
    clf_adv = deepcopy(clf)


    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]

        weights_adv, bias_adv = calTheta(x_r, weights_0, bias_0, alpha, theta_adv_method)
        clf_adv.model.coef_ = weights_adv.reshape(1,-1)
        clf_adv.model.intercept_ = bias_adv

        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [10]:
def runCostValidityTradeoff (results: dict, params: dict):
    for algorithm in params['algorithms']:
        for seed in params['seeds']:
            for v_alpha in params['alphas']:
                for v_lamb in params['lambdas']:
                    data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                    alpha = data["alpha"].unique().item()
                    lamb = data["lambda"].unique().item()
                    theta_0 = data["theta_0"].iloc[0]
                    X_0 = np.stack(data["x_0"])
                    X_r = np.stack(data["x_r"])

                    match params['adv_method']:
                        case "ONE":
                            res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "MANY":
                            res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "LARGESTALPHA":
                            res = evaluate_performance(X_0, X_r, theta_0, max(params['alphas']), lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "SMALLESTALPHA":
                            res = evaluate_performance(X_0, X_r, theta_0, min(params['alphas']), lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "THETA0":
                            res = evaluate_performance(X_0, X_r, theta_0, 0, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "_":
                            print(f"{params['adv_method']} does not exist!")

                    results['algorithm'].append(algorithm)
                    results['seed'].append(seed)
                    results['alpha'].append(alpha)
                    results['lambda'].append(lamb)
                    results['Cost'].append(res['cost'])
                    results['Current Validity'].append(res['m1_probability'])
                    results['Worst Case Validity'].append(res['wc_probability'])
                    # results['Current Validity'].append(res['m1_validity'])
                    # results['Worst Case Validity'].append(res['wc_validity'])
                    results['BCE Loss'].append(res['loss'])
                    results['J'].append(res['J'])


    if params['include_base_model']:
        for algo in ["BaseLineLInf", "BaseLineL1"]:
            algorithm = params['algorithms'][0]
            for seed in params['seeds']:
                for v_alpha in params['alphas']:
                    v_lamb = params['lambdas'][0]
                    data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")

                    alpha = data["alpha"].unique().item()
                    lamb = 0
                    theta_0 = data["theta_0"].iloc[0]
                    X_0 = np.stack(data["x_0"])
                    X_r = np.stack(data["x_0"])

                    match params['adv_method']:
                        case "ONE":
                            res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
                        case "MANY":
                            res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
                        case "LARGESTALPHA":
                            res = evaluate_performance(X_0, X_r, theta_0, max(params['alphas']), lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
                        case "SMALLESTALPHA":
                            res = evaluate_performance(X_0, X_r, theta_0, min(params['alphas']), lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
                        case "THETA0":
                            res = evaluate_performance(X_0, X_r, theta_0, 0, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
                        case "_":
                            print(f"{params['adv_method']} does not exist!")
                            
                    results['algorithm'].append(algo)
                    results['seed'].append(seed)
                    results['alpha'].append(alpha)
                    results['lambda'].append(lamb)
                    results['Cost'].append(res['cost'])
                    results['Current Validity'].append(res['m1_probability'])
                    results['Worst Case Validity'].append(res['wc_probability'])
                    results['BCE Loss'].append(res['loss'])
                    results['J'].append(res['J'])

    df_results = pd.DataFrame(results)
    return df_results

In [18]:
doAll = False

params = {}
params['alphas'] = [0.1]
params['lambdas'] = [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]
# params['lambdas'] = [0.1, 0.7, 1.4]
# params['lambdas'] = [2.1]
# 'lr', 'nn'
params['base_model'] = 'lr'
# 'synthetic', 'german', 'sba'
params['data'] = 'sba'
params['seeds'] = range(5)
# 'Alg1', 'L1PSD', 'ROARLInf', 'ROARL1'
params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
# params['algorithms'] = ['Alg1','L1PSD']
# 'ONE', 'MANY', 'LARGESTALPHA', 'SMALLESTALPHA', 'THETA0'
params['adv_method'] = 'LARGESTALPHA'
params['include_base_model'] = False

# dict_tmp = {params['algorithms'][0] : np.arange(0.02, 0.11, 0.02).round(4), 
#             params['algorithms'][1] : [0.02, 0.1], 
#             params['algorithms'][2] : np.arange(0.02, 0.11, 0.02).round(4), 
#             params['algorithms'][3] : np.arange(0.02, 0.11, 0.02).round(4)}

results = {
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
    'BCE Loss': [],
    'J': []
}

df_results_list = []
if doAll:
    adv_methods = ['MANY', 'LARGESTALPHA', 'SMALLESTALPHA', 'THETA0']
    for adv_method in adv_methods:
        params['adv_method'] = adv_method
        df_results_list.append(runCostValidityTradeoff(results, params))
else:
    df_results = runCostValidityTradeoff(results, params)

[Alg1] [ seed=0 ] [ α=0.1 ] [ λ=0.001 ]: 100%|██████████| 39/39 [00:00<00:00, 1904.73it/s]


[Roarl1] [ seed=4 ] [ α=0.1 ] [ λ=3.5 ]: 100%|██████████| 38/38 [00:00<00:00, 2621.27it/s]


In [19]:
print(f'{params["data"]}  |  {params["base_model"].upper()}')
df_results_avg = df_results.groupby(['algorithm', 'lambda'], as_index=False).mean(True)
df_results_im = df_results_avg.copy()
df_results_im[['Cost', 'Current Validity', 'Worst Case Validity', 'J']] = df_results_im[['Cost', 'Current Validity', 'Worst Case Validity', 'J']].round(2).astype(str) + '±' + df_results.groupby(['algorithm', 'lambda'], as_index=False).std(numeric_only=True)[['Cost', 'Current Validity', 'Worst Case Validity', 'J']].round(2).astype(str)

df_results_im

sba  |  LR


,algorithm,lambda,seed,alpha,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,Alg1,0.001,2.0,0.1,4.29±0.11,1.0±0.0,1.0±0.0,0.000234,0.0±0.0
1,Alg1,0.010,2.0,0.1,3.57±0.13,1.0±0.0,1.0±0.0,0.002342,0.04±0.0
2,Alg1,0.100,2.0,0.1,2.84±0.14,0.99±0.0,0.98±0.0,0.023765,0.31±0.01
3,Alg1,0.700,2.0,0.1,2.2±0.15,0.94±0.0,0.84±0.01,0.175486,1.72±0.1
4,Alg1,1.400,2.0,0.1,1.91±0.16,0.86±0.01,0.67±0.02,0.413298,3.08±0.21
5,Alg1,2.100,2.0,0.1,1.66±0.18,0.74±0.02,0.47±0.03,0.773137,4.26±0.33
6,Alg1,2.800,2.0,0.1,1.34±0.2,0.51±0.04,0.25±0.04,1.463306,5.22±0.45
7,Alg1,3.500,2.0,0.1,0.14±0.3,0.04±0.02,0.01±0.01,5.270487,5.74±0.65
8,L1PSD,0.001,2.0,0.1,4.24±0.5,1.0±0.0,1.0±0.0,0.000359,0.0±0.0
9,L1PSD,0.010,2.0,0.1,3.17±0.46,1.0±0.0,1.0±0.0,0.002812,0.03±0.0


In [20]:
if doAll:
    df_results_avg_list = []
    for df_results in df_results_list:
        df_results_avg_list.append(df_results.groupby(['algorithm', 'lambda', 'alpha'], as_index=False).mean())
else:
    df_results_avg = df_results.groupby(['algorithm', 'lambda', 'alpha'], as_index=False).mean()

df_results_avg

,algorithm,lambda,alpha,seed,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,Alg1,0.001,0.1,2.0,4.289407,0.999939,0.999766,0.000234,0.004523
1,Alg1,0.010,0.1,2.0,3.569565,0.999347,0.997661,0.002342,0.038037
2,Alg1,0.100,0.1,2.0,2.842503,0.992843,0.976545,0.023765,0.308015
3,Alg1,0.700,0.1,2.0,2.199815,0.943563,0.840423,0.175486,1.715356
4,Alg1,1.400,0.1,2.0,1.907114,0.864733,0.667015,0.413298,3.083257
5,Alg1,2.100,0.1,2.0,1.659141,0.739810,0.473831,0.773137,4.257333
6,Alg1,2.800,0.1,2.0,1.343082,0.505427,0.250817,1.463306,5.223937
7,Alg1,3.500,0.1,2.0,0.135235,0.037400,0.013124,5.270487,5.743808
8,L1PSD,0.001,0.1,2.0,4.243727,0.999690,0.999641,0.000359,0.004602
9,L1PSD,0.010,0.1,2.0,3.166683,0.997524,0.997192,0.002812,0.034479


In [21]:
df_results_avg = df_results.groupby(['algorithm', 'alpha', 'lambda'], as_index=False).mean()

In [22]:
# custom_colors = {
#     "LInf(Lamb = 0.1)": "#33FFFF",
#     "L1PSD(Lamb = 0.1)": "#FF3333",
#     "ROARL1(Lamb = 0.1)": "#33FF33",
#     "ROARLInf(Lamb = 0.1)": "#FF33FF",
#     "LInf(Lamb = 0.3)": "#33FFFF",
#     "L1PSD(Lamb = 0.3)": "#FF3333",
#     "ROARL1(Lamb = 0.3)": "#33FF33",
#     "ROARLInf(Lamb = 0.3)": "#FF33FF"
# }

# custom_colors = {
#     "LInf(Lamb = 0.1)": "#33FFFF",
#     "L1PSD(Lamb = 0.1)": "#FF3333",
#     "ROARL1(Lamb = 0.1)": "#33FF33",
#     "ROARLInf(Lamb = 0.1)": "#FF33FF",
#     "LInf(Lamb = 0.2)": "#33FFFF",
#     "L1PSD(Lamb = 0.2)": "#FF3333",
#     "ROARL1(Lamb = 0.2)": "#33FF33",
#     "ROARLInf(Lamb = 0.2)": "#FF33FF"
# }

data_map = {'synthetic': 'Synthetic', 'sba': 'Small Business Administration', 'german': 'German', 'income': 'ACS Income'}
model_map = {'lr': 'Logistic Regression', 'nn': 'Neural Network'}

# custom_colors = ["#33FFFF","#C8EFEF", "#FF3333", "#A71616", "#33FF33", "#207D20","#FF33FF", "#F7DAF7"]
custom_colors = ["#33FFFF", "#FF3333", "#33FF33","#FF33FF"]

# colors = ['#1f77b4', '#17becf', '#9467bd', '#e377c2', '#2ca02c'] # Synthesis
colors = ['#C7E8F0', "#7FCBDC", "#37AEC8", '#236F80', '#E2C2F4', "#BD74E7", "#9726D9", "#61188B"] # Synthesis
# colors = ['#17becf', '#e377c2', '#2ca02c'] # German
# colors = ['#17becf', '#9467bd', '#e377c2', '#2ca02c'] # SBA
nc = len(colors)

# fig = px.line(df_graph, 
#            x="Cost", y="Worst Case Validity", 
#            color="algorithm_lamb",
#            hover_data=["alpha", "lambda"],
#            title=f"{params['base_model']}_{params['data']}_{params['adv_method']}Adv",
#            markers=True,
#            color_discrete_map=custom_colors,
#            facet_col="lambda") 
# fig

In [23]:
def plotFigures(df_results_avg, params, should_plot_frontier=False):
    font_family = 'Times New Roman'
    font_color = 'black'
    width, height = 720, 540

    # symbols = ['x' for _ in range(len(params['lambdas']))] + ['circle']
    # size = [7 for _ in range(len(params['lambdas']))] + [5]

    fig = go.Figure()

    if should_plot_frontier:
        for i, alg in enumerate(params["algorithms"]):
            df_alg = df_results_avg.copy()
            df_alg = df_alg[(df_alg['algorithm']==alg)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
            x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
            df_alg = pd.DataFrame({'Algorithm': [f"{alg}" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask], 'lambda': df_alg['lambda'][mask]})

            fig.add_trace(go.Scatter(
                x = df_alg['Cost'],
                y = df_alg['Worst Case Validity'],
                mode = 'lines+markers' if alg != 'wachter' else 'markers',
                name = f"{alg}",
                showlegend=True,
                # customdata=df_alg['alpha'],
                customdata=df_alg[['alpha', 'lambda']].to_numpy(),
                hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata[0]}<br>lambda: %{customdata[1]}'
            ))

        if params['include_base_model']:
            for algo in ["BaseLineL1", "BaseLineLInf"]:
                df_alg = df_results_avg.copy()
                df_alg = df_alg[(df_alg['algorithm']==algo)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
                x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
                df_alg = pd.DataFrame({'Algorithm': [f"{algo}" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask], 'lambda': df_alg['lambda'][mask]})

                baseline_y = df_alg['Worst Case Validity'].mean()
                fig.add_trace(go.Scatter(
                    x=[0, df_results_avg['Cost'].max()],
                    y=[baseline_y, baseline_y],
                    mode="lines",
                    line=dict(dash="dash"),
                    name=f"{algo}",
                    showlegend=True
                ))
    else:
        c = 0
        
        # for i, alg in enumerate(params["algorithms"]):
        #     for lamb in params['lambdas']:
        #         df_alg = df_results_avg.copy()
        #         df_alg = df_alg[(df_alg['algorithm']==alg) & (df_alg['lambda']==lamb)]
        #         x, y = df_alg['Cost'], df_alg['Worst Case Validity']
        #         df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha']})

        #         # df_alg = df_alg[(df_alg['algorithm']==alg)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
        #         # x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
        #         # df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask]})

        #         fig.add_trace(go.Scatter(
        #             x = df_alg['Cost'],
        #             y = df_alg['Worst Case Validity'],
        #             # marker = dict(color=colors[c], size=5),
        #             # marker = dict(color=custom_colors[c], size=5),
        #             mode = 'lines+markers' if alg != 'wachter' else 'markers',
        #             name = f"{alg} (λ={lamb})",
        #             showlegend=True,
        #             customdata=df_alg['alpha'],
        #             hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata}'
        #         ))
        #         c+=1
            
        for i, alg in enumerate(params["algorithms"]):
            for alpha in params['alphas']:
                df_alg = df_results_avg.copy()
                df_alg = df_alg[(df_alg['algorithm']==alg) & (df_alg['alpha']==alpha)]
                x, y = df_alg['Cost'], df_alg['Worst Case Validity']
                df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(alpha={alpha})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'lambda': df_alg['lambda']})

                fig.add_trace(go.Scatter(
                    x = df_alg['Cost'],
                    y = df_alg['Worst Case Validity'],
                    # marker = dict(color=colors[c], size=5),
                    # marker = dict(color=custom_colors[c], size=5),
                    mode = 'lines+markers' if alg != 'wachter' else 'markers',
                    name = f"{alg} (alpha={alpha})",
                    showlegend=True,
                    customdata=df_alg['lambda'],
                    hovertemplate='Cost: %{x}<br>Validity: %{y}<br>lambda: %{customdata}'
                ))
                c+=1


        if params['include_base_model']:
            for algo in ["BaseLineL1", "BaseLineLInf"]:
                df_alg = df_results_avg.copy()
                df_alg = df_alg[(df_alg['algorithm']==algo) & (df_alg['lambda']==0)]
                x, y = df_alg['Cost'], df_alg['Worst Case Validity']
                df_alg = pd.DataFrame({'Algorithm': [f"{algo}_(λ=0)" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha']})

                baseline_y = df_alg['Worst Case Validity'].mean()
                fig.add_trace(go.Scatter(
                    x=[0, df_results_avg['Cost'].max()],
                    y=[baseline_y, baseline_y],
                    mode="lines",
                    line=dict(dash="dash"),
                    name=f"{algo}",
                    showlegend=True
                ))

    fig.update_xaxes(
        title=dict(
            text='Cost',
            font=dict(
                family=font_family,
                color=font_color,
                size=25
            )
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey', 
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )


    fig.update_yaxes(
        title=dict(
            text='Worst Case Validity',
            font=dict(
                family=font_family,
                color=font_color,
                size=25
            ), 
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey',
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )


    fig.update_layout(
        width=width,
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=50,b=25,l=25,r=25),
        title =dict(
            # text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | Average Adversary", 
            text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC ({params['adv_method']}) Adversary", 
            x= 0.5, 
            font=dict(family=font_family, size=20)
            ),
        legend=dict(
            x=0.975, 
            y=0.025, 
            orientation='v',
            xanchor='right',
            font=dict(
                family=font_family,
                color=font_color,
                size=15
                ), 
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='lightgrey',
            borderwidth=1,
            entrywidth=100.5,
            ),
        xaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20,
            ),
        ),
        yaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20
            ),
            range=[-0.1,1.1],
        )
    )

    return fig

In [24]:
should_plot_frontier = False

# font_family = 'Times New Roman'
# font_color = 'black'
# width, height = 720, 540

# symbols = ['x' for _ in range(len(params['lambdas']))] + ['circle']
# size = [7 for _ in range(len(params['lambdas']))] + [5]

# fig = go.Figure()

# if should_plot_frontier:
#     for i, alg in enumerate(params["algorithms"]):
#         df_alg = df_results_avg.copy()
#         df_alg = df_alg[(df_alg['algorithm']==alg)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
#         x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
#         df_alg = pd.DataFrame({'Algorithm': [f"{alg}" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask], 'lambda': df_alg['lambda'][mask]})

#         fig.add_trace(go.Scatter(
#             x = df_alg['Cost'],
#             y = df_alg['Worst Case Validity'],
#             mode = 'lines+markers' if alg != 'wachter' else 'markers',
#             name = f"{alg}",
#             showlegend=True,
#             # customdata=df_alg['alpha'],
#             customdata=df_alg[['alpha', 'lambda']].to_numpy(),
#             hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata[0]}<br>lambda: %{customdata[1]}'
#         ))

#     if params['include_base_model']:
#         for algo in ["BaseLineL1", "BaseLineLInf"]:
#             df_alg = df_results_avg.copy()
#             df_alg = df_alg[(df_alg['algorithm']==algo)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
#             x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
#             df_alg = pd.DataFrame({'Algorithm': [f"{algo}" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask], 'lambda': df_alg['lambda'][mask]})

#             baseline_y = df_alg['Worst Case Validity'].mean()
#             fig.add_trace(go.Scatter(
#                 x=[0, df_results_avg['Cost'].max()],
#                 y=[baseline_y, baseline_y],
#                 mode="lines",
#                 line=dict(dash="dash"),
#                 name=f"{algo}",
#                 showlegend=True
#             ))
# else:
#     c = 0
#     for i, alg in enumerate(params["algorithms"]):
#         for lamb in params['lambdas']:
#             # df_alg = df_results[(df_results['algorithm'] == alg) & (df_results['lambda']==lamb) & (df_results['alpha']<=0.2)].sort_values(['Cost'], ascending=True).copy().reset_index(drop=True)

#             df_alg = df_results_avg.copy()

#             df_alg = df_alg[(df_alg['algorithm']==alg) & (df_alg['lambda']==lamb)]
#             x, y = df_alg['Cost'], df_alg['Worst Case Validity']
#             df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha']})

#             # df_alg = df_alg[(df_alg['algorithm']==alg)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
#             # x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
#             # df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask]})

#             fig.add_trace(go.Scatter(
#                 x = df_alg['Cost'],
#                 y = df_alg['Worst Case Validity'],
#                 # marker = dict(color=colors[c], size=5),
#                 # marker = dict(color=custom_colors[c], size=5),
#                 mode = 'lines+markers' if alg != 'wachter' else 'markers',
#                 name = f"{alg} (λ={lamb})",
#                 showlegend=True,
#                 customdata=df_alg['alpha'],
#                 hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata}'
#             ))
#             c+=1

#     if params['include_base_model']:
#         for algo in ["BaseLineL1", "BaseLineLInf"]:
#             df_alg = df_results_avg.copy()
#             df_alg = df_alg[(df_alg['algorithm']==algo) & (df_alg['lambda']==0)]
#             x, y = df_alg['Cost'], df_alg['Worst Case Validity']
#             df_alg = pd.DataFrame({'Algorithm': [f"{algo}_(λ=0)" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha']})

#             baseline_y = df_alg['Worst Case Validity'].mean()
#             fig.add_trace(go.Scatter(
#                 x=[0, df_results_avg['Cost'].max()],
#                 y=[baseline_y, baseline_y],
#                 mode="lines",
#                 line=dict(dash="dash"),
#                 name=f"{algo}",
#                 showlegend=True
#             ))

# fig.update_xaxes(
#     title=dict(
#         text='Cost',
#         font=dict(
#             family=font_family,
#             color=font_color,
#             size=25
#         )
#         ), 
#     showline=True, 
#     mirror=True,
#     linecolor='black', 
#     gridcolor='lightgrey', 
#     zerolinewidth=1,
#     zerolinecolor='lightgrey',
#     )


# fig.update_yaxes(
#     title=dict(
#         text='Worst Case Validity',
#         font=dict(
#             family=font_family,
#             color=font_color,
#             size=25
#         ), 
#         ), 
#     showline=True, 
#     mirror=True,
#     linecolor='black', 
#     gridcolor='lightgrey',
#     zerolinewidth=1,
#     zerolinecolor='lightgrey',
#     )


# fig.update_layout(
#     width=width,
#     height=height,
#     plot_bgcolor='white',
#     paper_bgcolor='white',
#     margin=dict(t=50,b=25,l=25,r=25),
#     title =dict(
#         # text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | Average Adversary", 
#         text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC ({params['adv_method']}) Adversary", 
#         x= 0.5, 
#         font=dict(family=font_family, size=20)
#         ),
#     legend=dict(
#         x=0.975, 
#         y=0.025, 
#         orientation='v',
#         xanchor='right',
#         font=dict(
#             family=font_family,
#             color=font_color,
#             size=15
#             ), 
#         bgcolor='rgba(255, 255, 255, 0.7)',
#         bordercolor='lightgrey',
#         borderwidth=1,
#         entrywidth=100.5,
#         ),
#     xaxis=dict(
#         tickfont=dict(
#             family=font_family,
#             color=font_color,
#             size=20,
#         ),
#     ),
#     yaxis=dict(
#         tickfont=dict(
#             family=font_family,
#             color=font_color,
#             size=20
#         ),
#         range=[-0.1,1.1],
#     )
# )

fig = plotFigures(df_results_avg, params, should_plot_frontier)

print(f'{params["data"]}  |  {params["base_model"].upper()}')
fig.show()

sba  |  LR


In [51]:
# if should_plot_frontier:
#     figName = f"{params['base_model']}_{params['data']}_{params['adv_method']}_frontiers" 
#     fig.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-02\\" + 
#       figName + f".html")
# else:
#       figName = f"{params['base_model']}_{params['data']}_{params['adv_method']}_{params['lambdas'][0]}" 
#       fig.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-02\\" + 
#       figName + f".html")